<a href="https://colab.research.google.com/github/yo-danny/goodread-fiction-classifier/blob/main/Goodreads_RNN_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classify a book into fiction or non-fiction

This notebook uses RNN (Recurrent Neural Networks) to classify books in fiction/non-fiction based on the description on Goodreads.

In [ ]:
!pip install langdetect

## Extracting data from Goodreads

Using BeautifulSoup to access the < td > tag in each row of the table we can get its URL

In [ ]:
from bs4 import BeautifulSoup
import urllib.request
import pandas as pd
import time
import os

from langdetect import detect
from requests import get

# Header for not getting blocked
hdr = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.11 (KHTML, like Gecko) Chrome/23.0.1271.64 Safari/537.11',
         'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
         'Referer': 'https://cssspritegenerator.com',
         'Accept-Charset': 'ISO-8859-1,utf-8;q=0.7,*;q=0.3',
         'Accept-Encoding': 'none',
         'Accept-Language': 'en-US,en;q=0.8'}

BASE_URL = "https://www.goodreads.com"
LIST_URL = "https://www.goodreads.com/list/show/1.Best_Books_Ever"

books = {"URL":[]}

# Num of pages this list has
num_pages = 557
init_page = 1

for i in range (init_page, num_pages):
    time.sleep(20) # time to page load
    print(f"Reading page {i}")
    list_page_url = f"{LIST_URL}?page={i}"
    response = get(list_page_url, headers=hdr)
    html_soup = BeautifulSoup(response.content, 'html.parser')
    book_table = html_soup.find('table', attrs={'class':'tableList js-dataTooltip'})
    book_containers = book_table.find_all('tr')
    books['URL'] += [BASE_URL+r.find('a', attrs={'class':'bookTitle'})['href'] for r in book_containers]
    if i%50 == 0:
      books_df=pd.DataFrame.from_dict(books)
      books_df.to_csv(r'/content/drive/MyDrive/Goodreads/books.csv')

Reading page 1
Reading page 2
Reading page 3
Reading page 4
Reading page 5
Reading page 6
Reading page 7
Reading page 8
Reading page 9
Reading page 10
Reading page 11
Reading page 12
Reading page 13
Reading page 14
Reading page 15
Reading page 16
Reading page 17
Reading page 18
Reading page 19
Reading page 20
Reading page 21
Reading page 22
Reading page 23
Reading page 24
Reading page 25
Reading page 26
Reading page 27
Reading page 28
Reading page 29
Reading page 30
Reading page 31
Reading page 32
Reading page 33
Reading page 34
Reading page 35
Reading page 36
Reading page 37
Reading page 38
Reading page 39
Reading page 40
Reading page 41
Reading page 42
Reading page 43
Reading page 44
Reading page 45
Reading page 46
Reading page 47
Reading page 48
Reading page 49
Reading page 50
Reading page 51
Reading page 52
Reading page 53
Reading page 54
Reading page 55
Reading page 56
Reading page 57
Reading page 58
Reading page 59
Reading page 60
Reading page 61
Reading page 62
Reading page 63
R

In [ ]:
if not os.path.exists(r'/content/drive/MyDrive/Goodreads/book_data.csv'):
  book_data=pd.DataFrame(columns= [
      'image_url',
      'book_title',
      'book_authors',
      'book_rating',
      'book_rating_count',
      'book_review_count',
      'book_desc',
      'book_format',
      'book_edition',
      'book_pages',
      'book_isbn',
      'genres'
  ])

book_rows = []
save_every = 10

for i in range(len(book_data), len(books)):
  time.sleep(20)
  try:
    book_URL = books.at[i, 'URL']
    book_page = get(book_URL, headers=hdr)
    book_soup = BeautifulSoup(book_page.content, 'html.parser')

    book = dict()

    # cover of the book
    image_url = book_soup.find('img', attrs={'id':'coverImage'})
    if image_url:
       book['image_url'] = image_url.attrs['src']
    else:
      book['image_url'] = ''

    # title of the book
    book_title = book_soup.find('h1', attrs={'id':'bookTitle'})
    if book_title:
      book['book_title'] = book_title.text.replace('\n','').strip()
    else:
      book['book_title'] = ''

    #Author(s) of the book
    book['book_authors']='|'.join([a.find('span', attrs={'itemprop':'name'}).text for a in book_soup.find_all('a', attrs={'class':'authorName'})])

    #Rating given by users on goodreads
    book_rating=book_soup.find('span', attrs={'itemprop':'ratingValue'})
    if book_rating:
        book['book_rating']=book_rating.text.replace('\n','').strip()
    else:
        book['book_rating']=''
    book['book_rating_count']=book_soup.find('meta', attrs={'itemprop':'ratingCount'})['content']

    #No. of reviews for the book
    book['book_review_count']=book_soup.find('meta', attrs={'itemprop':'reviewCount'})['content']

    #A short description of the book, usually found on the back or inside cover of the book. Also called a blurb
    book_desc=book_soup.find('div', attrs={'class':'readable stacked'})
    if book_desc:
        book['book_desc']=book_desc.find_all('span')[-1].text
    else:
        book['book_desc']=''

    #Format of the book, e.g, paperback, hardcover, Kindle edition, etc.
    book_format=book_soup.find('div', attrs={'id':'details'}).find('span', attrs={'itemprop':'bookFormat'})
    if book_format:
        book['book_format']=book_format.text
    else:
        book['book_format']=''

    #Edition of the book
    book_edition=book_soup.find('div', attrs={'id':'details'}).find('span', attrs={'itemprop':'bookEdition'})
    if book_edition:
        book['book_edition']=book_edition.text
    else:
        book['book_edition']=''

    #No. of pages in the book
    book_pages=book_soup.find('div', attrs={'id':'details'}).find('span', attrs={'itemprop':'numberOfPages'})
    if book_pages:
        book['book_pages']=book_pages.text
    else:
        book['book_pages']=''

    #ISBN code of the book
    book_isbn=book_soup.find('div', attrs={'id':'bookDataBox'}).find('span', attrs={'itemprop':'isbn'})
    if book_isbn:
        book['book_isbn']=book_isbn.text
    else:
        book['book_isbn']=''

    #List of genres that the book belongs to. User supplied data.
    genres_list=book_soup.find_all('a', attrs={'class':'actionLinkLite bookPageGenreLink'})
    book['genres']='|'.join([i.text for i in genres_list])
    book_rows.append(book)

    if i%save_every==0:
        book_data.append(pd.DataFrame.from_dict(book_rows)).to_csv(r'/content/drive/MyDrive/Goodreads/book_data.csv', index=False)
        book_rows=[]
  except:
    book_data.append(pd.DataFrame.from_dict(book_rows)).to_csv(r'/content/drive/MyDrive/Goodreads/book_data.csv', index=False)
    book_rows=[]

## EDA (Exploratory Data Analysis)



In [ ]:
book_data_train = pd.read_csv(r'/content/drive/MyDrive/Goodreads/book_data.csv')
book_data_train.head()

The genres on Goodreads are supplied by the users, this can led to multiple genres which could be duplicates or trivial.

In [ ]:
genre_counts = defaultdict(int)
for i in book_data.index:
  g = book_data.at[i, 'genres']
  if type(g)==str:
    for j in g.split('|'):
      genre_counts[j]+=1

print(len(genre_counts))

### Data Cleaning

Excluding rows that don't have any genre or description, or that don't have the tagged genres of "fiction" or "non fiction".

Removed the non-English books from the dataset.

Plus some descriptions had formatting issues.

In [ ]:
def genre_binarizer(genres):
  """
  Analyses the genres string passed as argument and classify into fiction or nonfiction
  """
  genre_list = genres.lower().split("|")
  if "fiction" in genre_list:
    return "fiction"
  elif "nonfiction" in genre_list:
    return "non-fiction"
  else:
    return "na"

def remove_non_english(df):
  """
  Removes records that have invalid descripstions from the dataset
  Input: dataframe
  Outout: cleaned dataframe
  """
  invalid_desc_indxs = []
  for i in df.index:
    try:
      a=detect(df.at[i,'book_desc'])
      if a!='en':
        invalid_desc_indxs.append(i)
    except:
      invalid_desc_indxs.append(i)
  df=df.drop(index=invalid_desc_indxs)
  return df

def add_space_case(desc):
  """
  Analyses the book description passed and inserts a space after a lowercase letter followed by a uppercase letter.
  """
  upd_desc = ''

  for i in range(len(desc)-1):
    upd_desc+=desc[i]
    if desc[i] in string.ascii_lowercase and desc[i+1] in string.ascii_uppercase:
      upd_desc+=' '
  upd_desc+=desc[-1]
  return upd_desc

def remove_punctuation(desc):
  """
  Modifies the book description passed to
  - insert spaces in place of punctuations
  - join apostrophe words to their parent words
  - insert spaces where lowercase is followed by uppercase
  """
  desc = add_space_case(desc)
  apostrophe_words = ["m","re","ve","ll","t","s","d"]

  desc = desc.lower()

  desc="".join([c if c in valid_chars else "" for c in desc])

  for a in apostrophe_words:
    desc = desc.replace(""+a+"",a+"")

  return desc

def df_cleaner(df):
  """
  Performs the functions above on the dataset
  """
  df = df[df.genres.notnull()]
  df = df[df.book_desc.notnull()]
  df = df[book_desc.str.strip().apply(len)>0]
  df['binary_genre'] = df['genres'].apply(genre_binarizer)
  df = df[df.genres!='na']
  df = remove_non_english(df)
  df["clean_desc"] = df.book_desc.apply(remove_punctuation)

  df.reset_index(drop=True, inplace=True)
  return df

book_data_train = df_cleaner(book_data_train)
book_data_train.head()

